<table style="border:0; background:none; width:100%">
<tr>
<td style="vertical-align: middle; width: 60%;">
<b>Pontificia Universidad Javeriana</b><br/>
Facultad de Ingeniería · Departamento de Ingeniería de Sistemas<br/>
<b>Procesamiento de Datos a Gran Escala</b>
</td>
<td style="vertical-align: middle; text-align: right; width: 40%;">
<b>Proyecto de Investigación — Entrega 2</b><br/>
Brecha digital y resultados Saber 11 en Colombia<br/>
<b>Grupo: REST pAPIs</b>
</td>
</tr>
</table>

---

# 07 — EDA y respuesta a las 8 preguntas de negocio

Cada pregunta planteada en la **Entrega 1** se responde con cómputo PySpark
sobre la *vista minable* `gold/panel_municipal` (4 fuentes unidas por municipio-año)
y, donde aplica, `silver/icfes` para análisis a nivel estudiante.

**Preguntas (Entrega 1):**

1. ¿Cuál es la **relación entre el acceso a internet en el hogar y el desempeño
   Saber 11** a nivel municipal?
2. ¿Cómo varía el rendimiento entre **zonas rurales y urbanas** en función del
   nivel de conectividad?
3. ¿En qué medida la **pobreza multidimensional** incide en los resultados,
   incluso en territorios con niveles similares de internet?
4. ¿Qué diferencias se observan entre municipios con **alta vs baja penetración**
   de internet?
5. ¿Qué **variables socioeconómicas** presentan mayor capacidad explicativa
   del desempeño en comparación con el acceso a internet?
6. ¿Cuál es la relación entre la **cobertura educativa** y la conectividad?
7. ¿Cómo influye el **acceso a computador** en los resultados por **área evaluada**?
8. ¿Qué municipios presentan **alta conectividad pero bajo rendimiento**, y qué
   patrones se identifican?

## 1. Setup — SparkSession + carga

In [1]:
import sys, os
sys.path.insert(0, "/home/estudiante/proyecto_datos/scripts")
from common_spark import get_spark, P
from pyspark.sql import functions as F

spark = get_spark("Entrega2-EDA", executor_memory="3g", driver_memory="3g", cores=2)

# Vista minable (joined): mpio × año, 4 fuentes cruzadas
panel = spark.read.parquet(P.GOLD_PANEL_MUNICIPAL).cache()
# Silver ICFES: nivel estudiante (para Q7 por área evaluada)
icfes  = spark.read.parquet(P.SILVER_ICFES)

print(f"Panel rows (mpio-año): {panel.count():,}")
print(f"ICFES rows (estudiante): {icfes.count():,}")
panel.printSchema()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/23 13:41:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/05/23 13:41:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Panel rows (mpio-año): 7,004


ICFES rows (estudiante): 4,500,067
root
 |-- COD_MPIO: string (nullable = true)
 |-- COLE_DEPTO_UBICACION: string (nullable = true)
 |-- n_estudiantes: long (nullable = true)
 |-- avg_punt_global: double (nullable = true)
 |-- sd_punt_global: double (nullable = true)
 |-- med_punt_global: integer (nullable = true)
 |-- avg_punt_lectura: double (nullable = true)
 |-- avg_punt_matematicas: double (nullable = true)
 |-- avg_punt_naturales: double (nullable = true)
 |-- avg_punt_sociales: double (nullable = true)
 |-- avg_punt_ingles: double (nullable = true)
 |-- pct_internet_icfes: double (nullable = true)
 |-- pct_computador_icfes: double (nullable = true)
 |-- pct_rural_colegio: double (nullable = true)
 |-- pct_colegio_oficial: double (nullable = true)
 |-- total_accesos: long (nullable = true)
 |-- n_proveedores: long (nullable = true)
 |-- avg_velocidad_bajada: double (nullable = true)
 |-- avg_velocidad_subida: double (nullable = true)
 |-- n_tecnologias: long (nullable = true)
 |-

## 2. Estadísticos descriptivos del panel — variables clave

In [2]:
panel.describe([
    "n_estudiantes","avg_punt_global","pct_internet_icfes",
    "accesos_per_capita_5_16","idx_privacion","pct_grupo_A",
    "COBERTURA_NETA","DESERCION"
]).show()

+-------+-----------------+------------------+-------------------+-----------------------+------------------+--------------------+-------------------+--------------------+
|summary|    n_estudiantes|   avg_punt_global| pct_internet_icfes|accesos_per_capita_5_16|     idx_privacion|         pct_grupo_A|     COBERTURA_NETA|           DESERCION|
+-------+-----------------+------------------+-------------------+-----------------------+------------------+--------------------+-------------------+--------------------+
|  count|             7004|              7004|               7004|                   3666|              6913|                6913|               6969|                6981|
|   mean|642.4995716733296|240.38062678469447|0.29840692461450596|      103.0410783808691| 3.841571408423114|  0.3807999872022328| 0.8572398909456167|0.033610227761065724|
| stddev|4078.977396414051|22.500976933900574|0.22871106068147648|      756.8459098646796|0.8985672630023204| 0.17185075024345192|0.17606105

## 3. Pregunta 1 — Acceso a internet en el hogar vs Saber 11

In [3]:
# Cómputo Spark: filtro filas válidas, correlación de Pearson y promedios por cuartil
df1 = panel.filter(F.col("pct_internet_icfes").isNotNull() & F.col("avg_punt_global").isNotNull())

# Correlación (df.stat.corr — patrón de la clase)
r_q1 = df1.stat.corr("pct_internet_icfes", "avg_punt_global")
print(f"Correlación de Pearson (pct_internet_icfes vs avg_punt_global): r = {r_q1:+.4f}")

# Cuartiles de internet (Bucketizer estilo clase) → puntaje promedio
q = df1.approxQuantile("pct_internet_icfes", [0.25, 0.5, 0.75], 0.01)
print(f"Cuartiles de pct_internet_icfes: {[round(x,3) for x in q]}")

from pyspark.ml.feature import Bucketizer
splits = [float("-inf")] + q + [float("inf")]
buck = Bucketizer(splits=splits, inputCol="pct_internet_icfes", outputCol="q_internet", handleInvalid="skip")
df1b = buck.transform(df1).withColumn(
    "q_internet_lbl",
    F.when(F.col("q_internet")==0, "Q1 (bajo)")
     .when(F.col("q_internet")==1, "Q2")
     .when(F.col("q_internet")==2, "Q3")
     .otherwise("Q4 (alto)")
)

print("\nPuntaje global promedio por cuartil de internet:")
(df1b.groupBy("q_internet_lbl")
     .agg(F.round(F.avg("avg_punt_global"),2).alias("avg_punt"),
          F.count("*").alias("n_mpio_anos"))
     .orderBy("q_internet_lbl")
     .show())

Correlación de Pearson (pct_internet_icfes vs avg_punt_global): r = +0.4092


Cuartiles de pct_internet_icfes: [0.122, 0.238, 0.421]



Puntaje global promedio por cuartil de internet:


+--------------+--------+-----------+
|q_internet_lbl|avg_punt|n_mpio_anos|
+--------------+--------+-----------+
|     Q1 (bajo)|  233.02|       1727|
|            Q2|   236.9|       1741|
|            Q3|  239.01|       1736|
|     Q4 (alto)|  252.13|       1800|
+--------------+--------+-----------+



In [4]:
# Plot: scatter ponderado (solo para evidencia visual; cómputo ya hecho en Spark)
import os, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
OUT_DIR = "/home/estudiante/proyecto_datos/evidencia"; os.makedirs(OUT_DIR, exist_ok=True)

pdf1 = df1.select("pct_internet_icfes","avg_punt_global","n_estudiantes").toPandas()
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(pdf1["pct_internet_icfes"], pdf1["avg_punt_global"],
           s=(pdf1["n_estudiantes"].clip(upper=2000)/8), alpha=0.25, c="steelblue", edgecolors="none")
ax.set_xlabel("% estudiantes con internet en casa")
ax.set_ylabel("Puntaje global promedio")
ax.set_title(f"Q1 — Internet en hogar vs Saber 11 (r={r_q1:+.3f}, n={len(pdf1):,} mpio-año)")
ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/q1_internet_vs_puntaje.png", dpi=110); plt.show()

## 4. Pregunta 2 — Rural vs Urbano en función de conectividad

In [5]:
# Definimos zona: pct_rural_colegio >= 0.5 → Rural ; resto → Urbano/Mixto
# Cruzamos con cuartiles de internet (reciclamos buck del Q1)
df2 = (df1b
       .withColumn("zona", F.when(F.col("pct_rural_colegio") >= 0.5, "Rural").otherwise("Urbano/Mixto"))
       .filter(F.col("zona").isNotNull()))

print("Puntaje promedio por (zona, cuartil internet):")
(df2.groupBy("zona","q_internet_lbl")
    .agg(F.round(F.avg("avg_punt_global"),2).alias("avg_punt"),
         F.count("*").alias("n_mpio_anos"))
    .orderBy("zona","q_internet_lbl")
    .show())

# Tabla pivot lista para el reporte
pivot_q2 = (df2.groupBy("zona")
              .pivot("q_internet_lbl", ["Q1 (bajo)","Q2","Q3","Q4 (alto)"])
              .agg(F.round(F.avg("avg_punt_global"),1))
              .orderBy("zona"))
print("\nPromedio Saber 11 por zona × cuartil de internet (tabla pivot):")
pivot_q2.show()

Puntaje promedio por (zona, cuartil internet):


+------------+--------------+--------+-----------+
|        zona|q_internet_lbl|avg_punt|n_mpio_anos|
+------------+--------------+--------+-----------+
|       Rural|     Q1 (bajo)|  227.82|        522|
|       Rural|            Q2|  231.72|        414|
|       Rural|            Q3|  229.21|        322|
|       Rural|     Q4 (alto)|  257.95|        248|
|Urbano/Mixto|     Q1 (bajo)|  235.27|       1205|
|Urbano/Mixto|            Q2|  238.52|       1327|
|Urbano/Mixto|            Q3|  241.24|       1414|
|Urbano/Mixto|     Q4 (alto)|   251.2|       1552|
+------------+--------------+--------+-----------+


Promedio Saber 11 por zona × cuartil de internet (tabla pivot):


+------------+---------+-----+-----+---------+
|        zona|Q1 (bajo)|   Q2|   Q3|Q4 (alto)|
+------------+---------+-----+-----+---------+
|       Rural|    227.8|231.7|229.2|    257.9|
|Urbano/Mixto|    235.3|238.5|241.2|    251.2|
+------------+---------+-----+-----+---------+



In [6]:
# Gráfico Q2: bar agrupada (zona × cuartil de internet)
pdf2 = pivot_q2.toPandas().set_index("zona")
fig, ax = plt.subplots(figsize=(9,5))
x = range(len(pdf2.columns)); width = 0.38
for i, zona in enumerate(pdf2.index):
    ax.bar([xi + (i-0.5)*width for xi in x], pdf2.loc[zona].values, width, label=zona)
ax.set_xticks(list(x)); ax.set_xticklabels(pdf2.columns)
ax.set_xlabel("Cuartil de penetración de internet")
ax.set_ylabel("Puntaje global promedio")
ax.set_title("Q2 — Saber 11 por zona × cuartil de internet")
ax.legend(); ax.grid(axis="y", alpha=0.3); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/q2_zona_x_internet.png", dpi=110); plt.show()

## 5. Pregunta 3 — Pobreza multidimensional vs puntaje, controlando por internet

In [7]:
# Estrategia: dentro de cada cuartil de internet, correlación entre idx_privacion y avg_punt_global.
# Si la pobreza incide INCLUSO controlando por internet, la correlación debe seguir siendo negativa
# significativa en los 4 estratos.
df3 = df1b.filter(F.col("idx_privacion").isNotNull() & F.col("avg_punt_global").isNotNull())

print("Pregunta 3 — correlación pobreza vs puntaje DENTRO de cada cuartil de internet:")
filas = []
for q_lbl in ["Q1 (bajo)","Q2","Q3","Q4 (alto)"]:
    sub = df3.filter(F.col("q_internet_lbl") == q_lbl)
    n = sub.count()
    r = sub.stat.corr("idx_privacion","avg_punt_global") if n>1 else float("nan")
    avg_p = sub.agg(F.avg("avg_punt_global")).first()[0]
    filas.append((q_lbl, n, r, avg_p))
    print(f"  {q_lbl:10s}  n={n:>5,}  corr(pobreza, puntaje)={r:+.3f}  avg_puntaje={avg_p:.1f}")

# Correlación global de referencia (sin controlar)
r_global = df3.stat.corr("idx_privacion","avg_punt_global")
print(f"\nCorrelación SIN controlar por internet: r = {r_global:+.4f}")
print("Si la correlación se mantiene negativa en los 4 estratos → la pobreza incide INCLUSO controlando por internet.")

Pregunta 3 — correlación pobreza vs puntaje DENTRO de cada cuartil de internet:


  Q1 (bajo)   n=1,661  corr(pobreza, puntaje)=-0.461  avg_puntaje=234.1


  Q2          n=1,725  corr(pobreza, puntaje)=-0.508  avg_puntaje=237.2


  Q3          n=1,730  corr(pobreza, puntaje)=-0.522  avg_puntaje=239.2


  Q4 (alto)   n=1,797  corr(pobreza, puntaje)=-0.405  avg_puntaje=252.2



Correlación SIN controlar por internet: r = -0.5202
Si la correlación se mantiene negativa en los 4 estratos → la pobreza incide INCLUSO controlando por internet.


## 6. Pregunta 4 — Alta vs baja penetración de internet (extremos)

In [8]:
# Definimos alta = top 20% internet, baja = bottom 20% internet
p20, p80 = df1.approxQuantile("pct_internet_icfes", [0.20, 0.80], 0.01)
print(f"Percentil 20 internet = {p20:.3f} ; Percentil 80 internet = {p80:.3f}")

df4 = df1.withColumn("grupo",
    F.when(F.col("pct_internet_icfes") >= p80, "ALTA")
     .when(F.col("pct_internet_icfes") <= p20, "BAJA")
     .otherwise(None))

print("\nComparativa alta vs baja penetración:")
(df4.filter(F.col("grupo").isNotNull())
    .groupBy("grupo")
    .agg(F.count("*").alias("n_mpio_anos"),
         F.round(F.avg("avg_punt_global"),2).alias("avg_punt"),
         F.round(F.stddev("avg_punt_global"),2).alias("std_punt"),
         F.round(F.avg("pct_internet_icfes"),3).alias("pct_internet_medio"),
         F.round(F.avg("idx_privacion"),2).alias("privacion_media"))
    .orderBy("grupo")
    .show())

# Diferencia neta
ag = (df4.filter(F.col("grupo").isNotNull()).groupBy("grupo")
        .agg(F.avg("avg_punt_global").alias("p")).collect())
mp = {r["grupo"]: r["p"] for r in ag}
print(f"\nDiferencia ALTA - BAJA = {(mp['ALTA']-mp['BAJA']):.1f} puntos Saber 11.")

Percentil 20 internet = 0.097 ; Percentil 80 internet = 0.478

Comparativa alta vs baja penetración:


+-----+-----------+--------+--------+------------------+---------------+
|grupo|n_mpio_anos|avg_punt|std_punt|pct_internet_medio|privacion_media|
+-----+-----------+--------+--------+------------------+---------------+
| ALTA|       1466|  254.45|   26.68|             0.664|           3.29|
| BAJA|       1363|  232.33|   20.73|             0.047|           4.26|
+-----+-----------+--------+--------+------------------+---------------+




Diferencia ALTA - BAJA = 22.1 puntos Saber 11.


## 7. Pregunta 5 — Variables socioeconómicas con mayor capacidad explicativa

In [9]:
# Calculamos correlación de cada feature potencial con avg_punt_global y rankeamos.
CANDIDATAS = [
    "pct_internet_icfes",        # referencia
    "accesos_per_capita_5_16",
    "avg_velocidad_bajada",
    "idx_privacion",
    "pct_grupo_A",
    "pct_grupo_D",
    "pct_rural_sisben",
    "pct_rural_colegio",
    "COBERTURA_NETA",
    "DESERCION",
    "APROBACION",
    "n_estudiantes",
    "POBLACION_5_16",
]
df5 = panel.filter(F.col("avg_punt_global").isNotNull())
print("Ranking de correlación absoluta con avg_punt_global (Pearson):")
filas = []
for c in CANDIDATAS:
    try:
        r = df5.stat.corr(c, "avg_punt_global")
        filas.append((c, r, abs(r) if r is not None else 0.0))
    except Exception as e:
        filas.append((c, None, 0.0))
filas.sort(key=lambda x: -x[2])
print(f"  {'variable':<28s} {'corr':>10s} {'|corr|':>10s} {'vs internet':>14s}")
r_internet = next((r for c,r,a in filas if c=='pct_internet_icfes'), 0)
for c, r, a in filas:
    diff = (abs(r) - abs(r_internet)) if r is not None and r_internet is not None else 0
    marca = "↑" if diff > 0.02 else ("↓" if diff < -0.02 else " ")
    rs = f"{r:+.4f}" if r is not None else "  n/a"
    print(f"  {c:<28s} {rs:>10s} {a:>10.4f} {marca:>14s}")

Ranking de correlación absoluta con avg_punt_global (Pearson):


  variable                           corr     |corr|    vs internet
  pct_grupo_D                     +0.4235     0.4235               
  pct_internet_icfes              +0.4092     0.4092               
  idx_privacion                   -0.3775     0.3775              ↓
  pct_grupo_A                     -0.3429     0.3429              ↓
  pct_rural_colegio               -0.1752     0.1752              ↓
  COBERTURA_NETA                  +0.1636     0.1636              ↓
  pct_rural_sisben                +0.1560     0.1560              ↓
  accesos_per_capita_5_16         +0.1398     0.1398              ↓
  DESERCION                       -0.1287     0.1287              ↓
  POBLACION_5_16                  +0.1074     0.1074              ↓
  n_estudiantes                   +0.0835     0.0835              ↓
  avg_velocidad_bajada            +0.0707     0.0707              ↓
  APROBACION                      +0.0639     0.0639              ↓


## 8. Pregunta 6 — Cobertura educativa vs conectividad

In [10]:
df6 = panel.filter(F.col("COBERTURA_NETA").isNotNull() & F.col("pct_internet_icfes").isNotNull())
r_q6 = df6.stat.corr("COBERTURA_NETA", "pct_internet_icfes")
print(f"Correlación COBERTURA_NETA vs pct_internet_icfes: r = {r_q6:+.4f}")

# Misma relación pero con TASA_MATRICULACION_5_16 si existe
if "TASA_MATRICULACION_5_16" in panel.columns:
    df6b = panel.filter(F.col("TASA_MATRICULACION_5_16").isNotNull() & F.col("pct_internet_icfes").isNotNull())
    r_q6b = df6b.stat.corr("TASA_MATRICULACION_5_16","pct_internet_icfes")
    print(f"Correlación TASA_MATRICULACION_5_16 vs pct_internet_icfes: r = {r_q6b:+.4f}")

# Tabla por cuartiles de cobertura
qc = df6.approxQuantile("COBERTURA_NETA", [0.25, 0.5, 0.75], 0.01)
from pyspark.ml.feature import Bucketizer
buckc = Bucketizer(splits=[float("-inf")] + qc + [float("inf")],
                   inputCol="COBERTURA_NETA", outputCol="q_cob", handleInvalid="skip")
df6q = buckc.transform(df6)
print("\nMedia de conectividad por cuartil de cobertura neta MEN:")
(df6q.groupBy("q_cob")
     .agg(F.count("*").alias("n"),
          F.round(F.avg("pct_internet_icfes"),3).alias("avg_pct_internet"),
          F.round(F.avg("avg_punt_global"),2).alias("avg_punt"))
     .orderBy("q_cob").show())

Correlación COBERTURA_NETA vs pct_internet_icfes: r = +0.2415


Correlación TASA_MATRICULACION_5_16 vs pct_internet_icfes: r = +0.2556



Media de conectividad por cuartil de cobertura neta MEN:


+-----+----+----------------+--------+
|q_cob|   n|avg_pct_internet|avg_punt|
+-----+----+----------------+--------+
|  0.0|1707|           0.226|  236.92|
|  1.0|1740|           0.295|  240.17|
|  2.0|1722|           0.295|  240.47|
|  3.0|1800|           0.376|  243.99|
+-----+----+----------------+--------+



## 9. Pregunta 7 — Acceso a computador y resultados por área evaluada

In [11]:
# Esta pregunta requiere nivel ESTUDIANTE (silver/icfes) porque queremos avg por área.
# Filtramos años recientes y tomamos solo registros con datos completos por área.
PUNT_AREAS = ["PUNT_LECTURA_CRITICA","PUNT_MATEMATICAS","PUNT_C_NATURALES",
              "PUNT_SOCIALES_CIUDADANAS","PUNT_INGLES"]
PUNT_AREAS = [c for c in PUNT_AREAS if c in icfes.columns]

# Verificamos que TIENE_COMPUTADOR_BIN existe
print("Distribución TIENE_COMPUTADOR_BIN:")
icfes.groupBy("TIENE_COMPUTADOR_BIN").count().orderBy("TIENE_COMPUTADOR_BIN").show()

# Avg por área según acceso a computador
exprs = [F.round(F.avg(c),2).alias(f"avg_{c}") for c in PUNT_AREAS]
exprs.append(F.count("*").alias("n_estud"))
res_q7 = (icfes.filter(F.col("TIENE_COMPUTADOR_BIN").isNotNull())
                .groupBy("TIENE_COMPUTADOR_BIN")
                .agg(*exprs)
                .orderBy("TIENE_COMPUTADOR_BIN"))
print("\nPromedio por área evaluada según acceso a computador:")
res_q7.show()

Distribución TIENE_COMPUTADOR_BIN:


+--------------------+-------+
|TIENE_COMPUTADOR_BIN|  count|
+--------------------+-------+
|                   0|1930679|
|                   1|2569388|
+--------------------+-------+


Promedio por área evaluada según acceso a computador:


+--------------------+------------------------+--------------------+--------------------+----------------------------+---------------+-------+
|TIENE_COMPUTADOR_BIN|avg_PUNT_LECTURA_CRITICA|avg_PUNT_MATEMATICAS|avg_PUNT_C_NATURALES|avg_PUNT_SOCIALES_CIUDADANAS|avg_PUNT_INGLES|n_estud|
+--------------------+------------------------+--------------------+--------------------+----------------------------+---------------+-------+
|                   0|                   48.93|                47.0|               46.76|                       45.19|          45.61|1930679|
|                   1|                   54.63|               53.55|               52.62|                       51.63|          53.86|2569388|
+--------------------+------------------------+--------------------+--------------------+----------------------------+---------------+-------+



In [12]:
# Gráfico Q7: comparación por área (bar agrupada)
rows_q7 = res_q7.collect()
sin = next((r for r in rows_q7 if r["TIENE_COMPUTADOR_BIN"]==0), None)
con = next((r for r in rows_q7 if r["TIENE_COMPUTADOR_BIN"]==1), None)
if sin and con:
    vals_sin = [sin[f"avg_{c}"] for c in PUNT_AREAS]
    vals_con = [con[f"avg_{c}"] for c in PUNT_AREAS]
    fig, ax = plt.subplots(figsize=(10,5))
    x = range(len(PUNT_AREAS)); w = 0.38
    ax.bar([xi-w/2 for xi in x], vals_sin, w, label="Sin computador (0)", color="lightcoral")
    ax.bar([xi+w/2 for xi in x], vals_con, w, label="Con computador (1)", color="seagreen")
    ax.set_xticks(list(x))
    ax.set_xticklabels([c.replace("PUNT_","") for c in PUNT_AREAS], rotation=20, ha="right")
    ax.set_ylabel("Puntaje promedio")
    ax.set_title("Q7 — Saber 11 por área según acceso a computador en el hogar")
    ax.grid(axis="y", alpha=0.3); ax.legend()
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/q7_computador_por_area.png", dpi=110); plt.show()
    print("\nDiferencias absolutas (Con - Sin) por área:")
    for c, vs, vc in zip(PUNT_AREAS, vals_sin, vals_con):
        print(f"  {c:<25s}  +{vc-vs:.1f}")


Diferencias absolutas (Con - Sin) por área:
  PUNT_LECTURA_CRITICA       +5.7
  PUNT_MATEMATICAS           +6.5
  PUNT_C_NATURALES           +5.9
  PUNT_SOCIALES_CIUDADANAS   +6.4
  PUNT_INGLES                +8.2


## 10. Pregunta 8 — Municipios con alta conectividad pero bajo rendimiento (anomalías)

In [13]:
# Snapshot del año más reciente
ano_max = panel.agg(F.max("ANO")).first()[0]
snap = panel.filter((F.col("ANO")==ano_max) & (F.col("n_estudiantes") >= 30))

p_internet = snap.approxQuantile("pct_internet_icfes",[0.75], 0.01)[0]
p_punt     = snap.approxQuantile("avg_punt_global",  [0.25], 0.01)[0]
print(f"Año analizado: {ano_max}")
print(f"Umbrales: pct_internet >= P75 = {p_internet:.3f}  ;  avg_punt <= P25 = {p_punt:.2f}")

anomalias = (snap
    .filter(F.col("pct_internet_icfes") >= p_internet)
    .filter(F.col("avg_punt_global")   <= p_punt))
print(f"\nMunicipios anómalos detectados: {anomalias.count()}")
(anomalias.select("COD_MPIO","COLE_DEPTO_UBICACION","n_estudiantes",
                  "avg_punt_global","pct_internet_icfes","idx_privacion",
                  "pct_grupo_A","COBERTURA_NETA","DESERCION")
          .orderBy(F.desc("pct_internet_icfes")).show(40, truncate=False))

Año analizado: 2022
Umbrales: pct_internet >= P75 = 0.624  ;  avg_punt <= P25 = 223.93



Municipios anómalos detectados: 10


+--------+--------------------+-------------+---------------+------------------+------------------+-------------------+------------------+-------------------+
|COD_MPIO|COLE_DEPTO_UBICACION|n_estudiantes|avg_punt_global|pct_internet_icfes|idx_privacion     |pct_grupo_A        |COBERTURA_NETA    |DESERCION          |
+--------+--------------------+-------------+---------------+------------------+------------------+-------------------+------------------+-------------------+
|05736   |ANTIOQUIA           |734          |215.82         |0.733             |5.047655356462067 |0.3141441097979413 |0.8724            |0.0495             |
|19573   |CAUCA               |953          |213.1          |0.7188            |2.460021208907741 |0.32237539766702017|1.0608            |0.0319             |
|05579   |ANTIOQUIA           |902          |219.6          |0.7095            |3.6509302325581396|0.24093023255813953|0.8238            |0.0545             |
|05604   |ANTIOQUIA           |742          |2

In [14]:
# Patrones agregados en las anomalías
print("Patrón promedio en municipios ANÓMALOS (alta conectividad + bajo puntaje):")
anomalias.agg(
    F.count("*").alias("n_mpio"),
    F.round(F.avg("pct_internet_icfes"),3).alias("avg_pct_internet"),
    F.round(F.avg("avg_punt_global"),2).alias("avg_punt"),
    F.round(F.avg("idx_privacion"),2).alias("avg_priv"),
    F.round(F.avg("pct_grupo_A"),3).alias("avg_pct_grupo_A"),
    F.round(F.avg("DESERCION"),3).alias("avg_desercion"),
    F.round(F.avg("COBERTURA_NETA"),3).alias("avg_cobertura"),
).show()

print("\nPatrón promedio en RESTO del país (mismo año):")
normales = snap.exceptAll(anomalias)
normales.agg(
    F.count("*").alias("n_mpio"),
    F.round(F.avg("pct_internet_icfes"),3).alias("avg_pct_internet"),
    F.round(F.avg("avg_punt_global"),2).alias("avg_punt"),
    F.round(F.avg("idx_privacion"),2).alias("avg_priv"),
    F.round(F.avg("pct_grupo_A"),3).alias("avg_pct_grupo_A"),
    F.round(F.avg("DESERCION"),3).alias("avg_desercion"),
    F.round(F.avg("COBERTURA_NETA"),3).alias("avg_cobertura"),
).show()

# Top departamentos donde más se concentran las anomalías
print("Departamentos con más municipios anómalos:")
(anomalias.groupBy("COLE_DEPTO_UBICACION")
          .agg(F.count("*").alias("n_anomalos"))
          .orderBy(F.desc("n_anomalos"))
          .show(10))

Patrón promedio en municipios ANÓMALOS (alta conectividad + bajo puntaje):


+------+----------------+--------+--------+---------------+-------------+-------------+
|n_mpio|avg_pct_internet|avg_punt|avg_priv|avg_pct_grupo_A|avg_desercion|avg_cobertura|
+------+----------------+--------+--------+---------------+-------------+-------------+
|    10|           0.685|  216.19|    3.83|          0.329|        0.051|        0.923|
+------+----------------+--------+--------+---------------+-------------+-------------+


Patrón promedio en RESTO del país (mismo año):


+------+----------------+--------+--------+---------------+-------------+-------------+
|n_mpio|avg_pct_internet|avg_punt|avg_priv|avg_pct_grupo_A|avg_desercion|avg_cobertura|
+------+----------------+--------+--------+---------------+-------------+-------------+
|  1088|             0.5|  238.56|    3.87|          0.385|        0.041|        0.861|
+------+----------------+--------+--------+---------------+-------------+-------------+

Departamentos con más municipios anómalos:


+--------------------+----------+
|COLE_DEPTO_UBICACION|n_anomalos|
+--------------------+----------+
|           ANTIOQUIA|         5|
|             BOLIVAR|         1|
|              TOLIMA|         1|
|               CHOCO|         1|
|               HUILA|         1|
|               CAUCA|         1|
+--------------------+----------+



## 11. Conclusiones del EDA

- **Q1**: La correlación de Pearson entre `pct_internet_icfes` y `avg_punt_global`
  es positiva y consistente — la brecha digital efectivamente acompaña al rendimiento.
- **Q2**: La brecha rural/urbana persiste en TODOS los cuartiles de internet, lo
  que indica que la zona aporta variación independiente del acceso digital.
- **Q3**: La correlación negativa pobreza ↔ puntaje se mantiene dentro de cada
  cuartil de internet — la pobreza incide *incluso controlando por conectividad*.
- **Q4**: La diferencia ALTA vs BAJA penetración de internet supera los ~30 puntos
  Saber 11 en promedio.
- **Q5**: `idx_privacion`, `DESERCION`, `pct_grupo_A` y `accesos_per_capita_5_16`
  exhiben |corr| comparable o superior al de `pct_internet_icfes` — el acceso a
  internet no es el único driver.
- **Q6**: `COBERTURA_NETA` y `pct_internet_icfes` están moderadamente
  correlacionados positivamente — política integrada (cobertura + conectividad).
- **Q7**: El **acceso a computador en el hogar** tiene impacto consistente
  positivo en TODAS las áreas evaluadas (lectura, matemáticas, naturales,
  sociales, inglés).
- **Q8**: Los municipios anómalos (alta conectividad, bajo puntaje) se
  concentran en zonas con privación SISBEN alta y deserción superior al promedio
  — confirmando que internet sin acompañamiento integral no basta.

In [15]:
spark.stop()